In [17]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

from langchain_huggingface import HuggingFaceEndpoint
from langchain_huggingface.chat_models import ChatHuggingFace
from dotenv import load_dotenv
import os



In [18]:
load_dotenv()

# Initialize HuggingFace LLM
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_KEY")
)

model = ChatHuggingFace(llm=llm)



In [19]:
# Create a State
class LLMState(TypedDict):
    
    question: str
    answer: str

In [20]:
def llm_question(state: LLMState) -> LLMState:

    #extract question from state
    question = state["question"]

    # from prompt
    prompt = f"Answer the following question concisely: {question}"

    #ask the question to the model
    answer = model.invoke(prompt).content
    
    # update the answer in the state
    state["answer"] = answer
    return state


In [22]:
# Create Graph
graph = StateGraph(LLMState)

# Add Nodes
graph.add_node("LLM_Question_Node", llm_question)

# define edges
graph.add_edge(START, "LLM_Question_Node")
graph.add_edge("LLM_Question_Node", END)

# Compile the graph
workflow = graph.compile()

# Execute the graph
initial_state = {
    "question": "What is the capital of France?",
    "answer": ""
}


final_state = workflow.invoke(initial_state)
print("Final State:", final_state['answer'])

Final State: Paris.
